# Structured Outputs — Practice

Companion to [../06_structured_outputs.md](../06_structured_outputs.md).

In [ ]:
#install dependent py modules
from anthropic import Anthropic
from dotenv import load_dotenv
from IPython.display import display, Markdown, clear_output
import json


load_dotenv()

In [ ]:
# Params
model = "claude-sonnet-5"
# model = "claude-haiku-4-5-20251001"

# SDK client
client = Anthropic()

In [ ]:
#helper functions
def add_user_message(messages, text):
    message = { "role": "user", "content": text}
    messages.append(message)

def add_assistant_message(messages, text):
    message = { "role": "assistant", "content": text}
    messages.append(message)

def chat(messages, system_prompt=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1024,
        "messages": messages,
        "stop_sequences": stop_sequences
    }

    if system_prompt:
        params["system"] = system_prompt

    response = client.messages.create(**params)
    text_blocks = [block.text for block in response.content if block.type == "text"]
    return "\n".join(text_blocks)

# Prettify claude md response
def display_turn(role, text):
    display(Markdown(f"**{role}:**\n\n{text}"))

In [ ]:
messages = []

add_user_message(messages, "Hello Sonnet! - Give me a dummy user details in json format for my user profile table?")
response_text = chat(messages)

display_turn("Assistant", response_text)

In [ ]:
# Why are we using a system prompt here? as the new models so not support prefilling to get a structured output of. your choice
# What prefill requries is a user message followed by a assistant message, and the new models do not allow assistant message to be the last one.
# Better approach is to use a system prompt which covers all the rules, claude needs to follow

system_prompt = """
You are an expert in writing json file and keeps them in well formatted to improve readability.

Whenever a user asks to return a json format output, you follow the bellow pattern, with no surrounding greetings or comments.
exmaple:
User prompts - 'Generate a json for a user role, this role defines what operations is can do'
Assistand response - 
{
    "user": {
        "id": 123,
        "firstName": "Billy",
        "lastName": "Shines"
    },
    "role": {
        "id": 6,
        "name": "admin"
    }

}

"""
messages = []

add_user_message(messages, "Hello Sonnet! - Give me a dummy user details in json format for my user profile table?")
response_text = chat(messages, system_prompt=system_prompt)

response_text


In [ ]:
json.loads(response_text)